# 03 · Data Formats: JSON, CSV, Pickle, TOML, INI & SQLite

**Module 11 – Files & OS** · Estimated time: 60 min

> 📗 The bootcamp covers CSV (in its PDFs & Spreadsheets section). **JSON**, the most common data format on the web, is missing from both sources, as are TOML, pickle and SQLite. All are included here.

### 🎯 What you will learn
- **JSON**: `dumps/loads/dump/load`, pretty printing, type mapping, custom objects
- **CSV**: `reader`/`writer`, ✅ `DictReader`/`DictWriter`, encodings, the `newline=""` rule
- **pickle**: saving Python objects (and its security warning)
- **TOML** (`tomllib`, 3.11+) and **INI** (`configparser`) config files
- **SQLite** (`sqlite3`): a real SQL database in one file

## Part 1: JSON

**JSON** (JavaScript Object Notation) is the universal format for APIs and config files. It maps naturally to Python:

| JSON | Python |
|---|---|
| object `{}` | `dict` |
| array `[]` | `list` |
| string | `str` |
| number | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

In [1]:
import json

user = {"name": "Asha", "age": 28, "skills": ["python", "sql"], "active": True, "manager": None}

text = json.dumps(user)                        # Python → JSON string
print(text, type(text))
back = json.loads(text)                        # JSON string → Python
print(back, type(back))
print(json.dumps(user, indent=2, sort_keys=True))     # pretty print

{"name": "Asha", "age": 28, "skills": ["python", "sql"], "active": true, "manager": null} <class 'str'>
{'name': 'Asha', 'age': 28, 'skills': ['python', 'sql'], 'active': True, 'manager': None} <class 'dict'>
{
  "active": true,
  "age": 28,
  "manager": null,
  "name": "Asha",
  "skills": [
    "python",
    "sql"
  ]
}


### Files: `dump` / `load` (no "s")

In [2]:
with open("user.json", "w", encoding="utf-8") as f:
    json.dump(user, f, indent=2, ensure_ascii=False)   # ensure_ascii=False keeps non-English text readable

with open("user.json", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded["skills"])

['python', 'sql']


### Things JSON can't store directly

In [3]:
from datetime import datetime
try:
    json.dumps({"when": datetime(2026, 10, 20)})
except TypeError as e:
    print("TypeError:", e)

# Fix: a `default` function converts unknown types
def encode(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Can't serialize {type(obj).__name__}")

print(json.dumps({"when": datetime(2026, 10, 20, 9, 30), "tags": {"b", "a"}}, default=encode))

TypeError: Object of type datetime is not JSON serializable
{"when": "2026-10-20T09:30:00", "tags": ["a", "b"]}


Note: JSON object keys are always strings, and tuples become lists:

In [4]:
print(json.loads(json.dumps({1: (1, 2)})))

{'1': [1, 2]}


### Dataclasses ↔ JSON

In [5]:
from dataclasses import dataclass, asdict

@dataclass
class Product:
    name: str
    price: float

items = [Product("Pen", 10.0), Product("Book", 250.5)]
payload = json.dumps([asdict(p) for p in items])
print(payload)
restored = [Product(**d) for d in json.loads(payload)]
print(restored)

[{"name": "Pen", "price": 10.0}, {"name": "Book", "price": 250.5}]
[Product(name='Pen', price=10.0), Product(name='Book', price=250.5)]


> 💡 `python -m json.tool file.json` pretty-prints JSON from the terminal. For validated JSON ↔ objects in larger apps, look at **Pydantic**.

## Part 2: CSV (📗 Working with CSV Files)

CSV = Comma-Separated Values: plain-text tables that Excel, Google Sheets and databases can read and write.

In [6]:
with open("example.csv", encoding="utf-8") as f:     # a course asset file
    print(f.read()[:300])

id,first_name,last_name,email,gender,ip_address,city
1,Joseph,Zaniolini,jzaniolini0@simplemachines.org,Male,163.168.68.132,Pedro Leopoldo
2,Freida,Drillingcourt,fdrillingcourt1@umich.edu,Female,97.212.102.79,Buri
3,Nanni,Herity,nherity2@statcounter.com,Female,145.151.178.98,Claver
4,Orazio,Frayling,


### Reading with `csv.reader`

In [7]:
import csv

with open("example.csv", encoding="utf-8", newline="") as f:
    csv_data = csv.reader(f)
    data_lines = list(csv_data)

print(data_lines[0])                 # header row
print(len(data_lines), "rows")
print(data_lines[10])
print(data_lines[10][3])             # a single cell: the email column

['id', 'first_name', 'last_name', 'email', 'gender', 'ip_address', 'city']
1001 rows
['10', 'Hyatt', 'Gasquoine', 'hgasquoine9@google.ru', 'Male', '221.155.106.39', 'Złoty Stok']
hgasquoine9@google.ru


In [8]:
# All emails
emails = [line[3] for line in data_lines[1:]]
print(emails[:5])
# Full names
full_names = [f"{line[1]} {line[2]}" for line in data_lines[1:]]
print(full_names[:5])

['jzaniolini0@simplemachines.org', 'fdrillingcourt1@umich.edu', 'nherity2@statcounter.com', 'ofrayling3@economist.com', 'jmurrison4@cbslocal.com']
['Joseph Zaniolini', 'Freida Drillingcourt', 'Nanni Herity', 'Orazio Frayling', 'Julianne Murrison']


### Encoding (📗)
CSV files may contain characters like accented letters or symbols. If you get a `UnicodeDecodeError`, the file is not UTF-8. Try `encoding="utf-8-sig"` (UTF-8 with a BOM, common from Excel) or `encoding="latin-1"`/`"cp1252"`.

> ⚠️ **Always pass `newline=""` when opening CSV files** (reading or writing). Otherwise you get blank lines between rows on Windows, and quoted fields with embedded newlines break.

### ✅ `DictReader`: rows as dictionaries (much more readable)

In [9]:
with open("example.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(reader.fieldnames)
print(rows[0])
print(rows[0]["email"])
cities = {r["city"] for r in rows}
print(len(cities), "different cities")

['id', 'first_name', 'last_name', 'email', 'gender', 'ip_address', 'city']
{'id': '1', 'first_name': 'Joseph', 'last_name': 'Zaniolini', 'email': 'jzaniolini0@simplemachines.org', 'gender': 'Male', 'ip_address': '163.168.68.132', 'city': 'Pedro Leopoldo'}
jzaniolini0@simplemachines.org
976 different cities


### Writing CSV files

In [10]:
with open("to_save_file.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter=",")
    writer.writerow(["a", "b", "c"])
    writer.writerows([["1", "2", "3"], ["4", "5", "6"]])
    writer.writerow(["has, comma", 'has "quotes"', "multi\nline"])    # handled by quoting

print(open("to_save_file.csv", encoding="utf-8").read())

a,b,c
1,2,3
4,5,6
"has, comma","has ""quotes""","multi
line"



### Appending to an existing file (📗)

In [11]:
with open("to_save_file.csv", "a", encoding="utf-8", newline="") as f:
    csv.writer(f).writerow(["new", "new", "new"])
print(open("to_save_file.csv", encoding="utf-8").read().splitlines()[-1])

new,new,new


### `DictWriter`

In [12]:
scores = [{"name": "Asha", "score": 91}, {"name": "Ravi", "score": 78}]
with open("scores.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerows(scores)
print(open("scores.csv", encoding="utf-8").read())

name,score
Asha,91
Ravi,78



> 💡 Values read from CSV are always **strings**: convert with `int()`/`float()`. For serious data analysis, use **pandas** (`pd.read_csv`), which handles types, missing values and much more.

## Part 3: pickle: saving arbitrary Python objects

In [13]:
import pickle

state = {"level": 3, "inventory": {"sword", "shield"}, "pos": (10, 20)}
with open("game.pkl", "wb") as f:          # binary mode!
    pickle.dump(state, f)
with open("game.pkl", "rb") as f:
    print(pickle.load(f))

{'level': 3, 'inventory': {'sword', 'shield'}, 'pos': (10, 20)}


> ⚠️ **Security warning:** **never unpickle data from an untrusted source.** A malicious pickle can run arbitrary code on your machine. pickle is Python-only and can break between versions of your classes. Prefer JSON for data exchange.

## Part 4: Config files: TOML and INI

### TOML (`tomllib`, 3.11+, read-only)

In [14]:
import tomllib

toml_text = """
title = "My App"
debug = false

[database]
host = "localhost"
port = 5432
replicas = ["db1", "db2"]
"""
with open("config.toml", "w", encoding="utf-8") as f:
    f.write(toml_text)

with open("config.toml", "rb") as f:        # tomllib requires binary mode
    config = tomllib.load(f)
print(config)
print(config["database"]["port"] + 1)        # real ints, bools and lists, not strings

{'title': 'My App', 'debug': False, 'database': {'host': 'localhost', 'port': 5432, 'replicas': ['db1', 'db2']}}
5433


(To *write* TOML, install the `tomli-w` package.)

### INI (`configparser`)

In [15]:
import configparser

cp = configparser.ConfigParser()
cp["server"] = {"host": "0.0.0.0", "port": "8000"}
cp["features"] = {"dark_mode": "yes"}
with open("settings.ini", "w", encoding="utf-8") as f:
    cp.write(f)

cp2 = configparser.ConfigParser()
cp2.read("settings.ini", encoding="utf-8")
print(cp2["server"]["host"], cp2.getint("server", "port"), cp2.getboolean("features", "dark_mode"))

0.0.0.0 8000 True


## Part 5: SQLite: a database in a single file

`sqlite3` is built into Python: no server needed. Great for apps, prototypes and learning SQL.

In [16]:
import sqlite3

with sqlite3.connect("shop.db") as conn:              # the with block commits on success
    conn.execute("DROP TABLE IF EXISTS products")
    conn.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)")
    conn.executemany("INSERT INTO products (name, price) VALUES (?, ?)",
                     [("Pen", 10), ("Notebook", 55.5), ("Bag", 899)])

conn = sqlite3.connect("shop.db")
conn.row_factory = sqlite3.Row                         # rows behave like dicts
for row in conn.execute("SELECT name, price FROM products WHERE price < ? ORDER BY price", (100,)):
    print(row["name"], row["price"])
print(conn.execute("SELECT COUNT(*), AVG(price) FROM products").fetchone()[:])
conn.close()

Pen 10.0
Notebook 55.5
(3, 321.5)


> ⚠️ **Always use `?` placeholders** for values, never f-strings, to prevent **SQL injection**:
> ```python
> conn.execute(f"SELECT * FROM users WHERE name = '{name}'")      # ❌ dangerous
> conn.execute("SELECT * FROM users WHERE name = ?", (name,))      # ✅ safe
> ```

## 📝 Summary

| Format | Module | Read | Write | Use for |
|---|---|---|---|---|
| JSON | `json` | `load(s)` | `dump(s)` | APIs, configs, data exchange |
| CSV | `csv` | `DictReader` | `DictWriter` | tables, spreadsheets |
| pickle | `pickle` | `load` | `dump` | Python-only caches (trusted data!) |
| TOML | `tomllib` | `load` | (`tomli-w`) | config, `pyproject.toml` |
| INI | `configparser` | `read` | `write` | legacy config |
| SQLite | `sqlite3` | `SELECT` | `INSERT` | structured, queryable data |

## 🏋️ Practice
1. Save a list of 3 book dicts (`title`, `author`, `year`) to `books.json`, load it back, and print the oldest book.
2. Convert `books.json` to `books.csv` with `DictWriter`.
3. Read `example.csv` and count how many people have an email ending in `.com`.
4. Create an SQLite table `books`, insert the 3 books, and query those published after 2000.

<details><summary>Show solutions</summary>

```python
import json, csv, sqlite3

books = [{"title": "Fluent Python", "author": "Ramalho", "year": 2022},
         {"title": "SICP", "author": "Abelson", "year": 1985},
         {"title": "Clean Code", "author": "Martin", "year": 2008}]
with open("books.json", "w", encoding="utf-8") as f:
    json.dump(books, f, indent=2)
with open("books.json", encoding="utf-8") as f:
    data = json.load(f)
print(min(data, key=lambda b: b["year"]))

with open("books.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["title", "author", "year"])
    w.writeheader()
    w.writerows(data)

with open("example.csv", encoding="utf-8", newline="") as f:
    print(sum(r["email"].endswith(".com") for r in csv.DictReader(f)))

with sqlite3.connect("books.db") as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS books (title TEXT, author TEXT, year INT)")
    conn.executemany("INSERT INTO books VALUES (:title, :author, :year)", data)
    print(conn.execute("SELECT title FROM books WHERE year > 2000").fetchall())
```
</details>